# MuJoCo Reacher Behavior Cloning Robustness Evaluation

This notebook is a fast, interview-ready robotics learning mini-project.

Goal: train a behavior cloning policy on `Reacher-v5`, then evaluate how brittle it is under realistic deployment issues: observation noise, actuator noise, latency, and dynamics mismatch.

Narrative hook: **real robots do not give policies clean observations, perfect actuation, or identical dynamics.** This connects robotics deployment experience to robot-learning generalization.

In [ ]:
# Colab setup. Runtime > Change runtime type > T4 GPU is optional; CPU is fine.
!pip -q install "gymnasium[mujoco]" torch pandas matplotlib tqdm

In [ ]:
import math, random, time
from dataclasses import dataclass

import gymnasium as gym
import matplotlib.pyplot as plt
import mujoco
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

## 1. Expert Controller

Instead of training a slow RL expert, we use a Jacobian-transpose controller as a simple demonstration source. This is fair for the project because the point is not to invent a perfect expert. The point is to study what happens when an imitation policy faces distribution shift.

In [ ]:
def site_id(model, name):
    return mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, name)

def expert_action(env, kp=12.0, kd=1.2):
    """Jacobian-transpose controller: action = J^T * position_error - damping."""
    unwrapped = env.unwrapped
    model, data = unwrapped.model, unwrapped.data
    fingertip = site_id(model, "fingertip")
    target = site_id(model, "target")

    err = data.site_xpos[target] - data.site_xpos[fingertip]
    jacp = np.zeros((3, model.nv))
    jacr = np.zeros((3, model.nv))
    mujoco.mj_jacSite(model, data, jacp, jacr, fingertip)

    torque = kp * jacp[:, :2].T @ err - kd * data.qvel[:2]
    return np.clip(torque, env.action_space.low, env.action_space.high).astype(np.float32)

def make_env(seed=SEED, damping_scale=1.0, mass_scale=1.0):
    env = gym.make("Reacher-v5")
    env.reset(seed=seed)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
    env.unwrapped.model.dof_damping[:] *= damping_scale
    env.unwrapped.model.body_mass[:] *= mass_scale
    return env

env = make_env()
obs, info = env.reset(seed=SEED)
obs.shape, env.action_space.shape, expert_action(env)

## 2. Collect Demonstrations

In [ ]:
def collect_demos(num_episodes=250, seed=SEED):
    env = make_env(seed=seed)
    obs_list, act_list, returns = [], [], []
    for ep in tqdm(range(num_episodes), desc="Collecting demos"):
        obs, _ = env.reset(seed=seed + ep)
        done = False
        total = 0.0
        while not done:
            act = expert_action(env)
            obs_list.append(obs.astype(np.float32))
            act_list.append(act.astype(np.float32))
            obs, reward, terminated, truncated, _ = env.step(act)
            done = terminated or truncated
            total += reward
        returns.append(total)
    env.close()
    return np.array(obs_list), np.array(act_list), np.array(returns)

X, y, expert_returns = collect_demos(num_episodes=250)
print(X.shape, y.shape)
print("Expert return mean/std:", expert_returns.mean(), expert_returns.std())

## 3. Train Behavior Cloning Policy

In [ ]:
class BCPolicy(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, act_dim), nn.Tanh()
        )
    def forward(self, x):
        return self.net(x)

obs_mean = X.mean(axis=0, keepdims=True)
obs_std = X.std(axis=0, keepdims=True) + 1e-6
Xn = (X - obs_mean) / obs_std

ds = TensorDataset(torch.tensor(Xn, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
dl = DataLoader(ds, batch_size=512, shuffle=True)

policy = BCPolicy(X.shape[1], y.shape[1]).to(DEVICE)
opt = torch.optim.AdamW(policy.parameters(), lr=3e-4, weight_decay=1e-4)
loss_fn = nn.MSELoss()

losses = []
for epoch in tqdm(range(35), desc="Training BC"):
    epoch_losses = []
    for xb, yb in dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = policy(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        epoch_losses.append(loss.item())
    losses.append(float(np.mean(epoch_losses)))

plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Behavior Cloning Training Loss")
plt.grid(True)
plt.show()

## 4. Robustness Evaluation

We evaluate return under four conditions:

- Clean environment
- Observation noise
- Actuator noise
- Latency from stale observations
- Dynamics mismatch by changing damping and mass

In [ ]:
@torch.no_grad()
def act_policy(obs):
    x = ((obs[None, :] - obs_mean) / obs_std).astype(np.float32)
    a = policy(torch.tensor(x, device=DEVICE)).cpu().numpy()[0]
    return np.clip(a, -1.0, 1.0).astype(np.float32)

def evaluate(condition_name, episodes=60, obs_noise=0.0, action_noise=0.0, latency_steps=0,
             damping_scale=1.0, mass_scale=1.0, seed=1000):
    env = make_env(seed=seed, damping_scale=damping_scale, mass_scale=mass_scale)
    returns, final_dists = [], []
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        obs_buffer = [obs.copy() for _ in range(latency_steps + 1)]
        done = False
        total = 0.0
        while not done:
            policy_obs = obs_buffer[0].copy()
            if obs_noise > 0:
                policy_obs = policy_obs + np.random.normal(0, obs_noise, size=policy_obs.shape)
            action = act_policy(policy_obs)
            if action_noise > 0:
                action = action + np.random.normal(0, action_noise, size=action.shape)
            action = np.clip(action, env.action_space.low, env.action_space.high).astype(np.float32)
            obs, reward, terminated, truncated, _ = env.step(action)
            obs_buffer.append(obs.copy())
            obs_buffer = obs_buffer[-(latency_steps + 1):]
            done = terminated or truncated
            total += reward

        fingertip = site_id(env.unwrapped.model, "fingertip")
        target = site_id(env.unwrapped.model, "target")
        dist = np.linalg.norm(env.unwrapped.data.site_xpos[target] - env.unwrapped.data.site_xpos[fingertip])
        returns.append(total)
        final_dists.append(dist)
    env.close()
    return {
        "condition": condition_name,
        "return_mean": np.mean(returns),
        "return_std": np.std(returns),
        "final_distance_mean": np.mean(final_dists),
        "success_rate_dist_lt_0.07": np.mean(np.array(final_dists) < 0.07),
    }

conditions = [
    ("clean", dict()),
    ("obs_noise_0.02", dict(obs_noise=0.02)),
    ("obs_noise_0.05", dict(obs_noise=0.05)),
    ("actuator_noise_0.10", dict(action_noise=0.10)),
    ("latency_2_steps", dict(latency_steps=2)),
    ("dynamics_mismatch", dict(damping_scale=1.8, mass_scale=1.25)),
]

rows = []
for name, kwargs in tqdm(conditions, desc="Evaluating"):
    rows.append(evaluate(name, **kwargs))

results = pd.DataFrame(rows)
results

In [ ]:
display(results.sort_values("return_mean", ascending=False))

plt.figure(figsize=(10, 4))
plt.bar(results["condition"], results["return_mean"], yerr=results["return_std"], capsize=4)
plt.xticks(rotation=30, ha="right")
plt.ylabel("Episode Return, higher is better")
plt.title("Behavior Cloning Policy Robustness on MuJoCo Reacher")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.bar(results["condition"], results["success_rate_dist_lt_0.07"])
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1)
plt.ylabel("Success Rate")
plt.title("Success Rate Under Deployment Perturbations")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Interview Talking Points

Use this as the discussion frame:

> I built a small behavior cloning setup in MuJoCo Reacher, but the part I cared about most was not just clean-environment performance. I evaluated how the cloned policy degrades under observation noise, actuator noise, latency, and dynamics mismatch, because those are the kinds of problems I have seen matter on physical robotics systems.

> The main takeaway was that imitation policies can look fine in clean conditions while being brittle under small deployment shifts. That made me more interested in representations and training procedures that improve robustness, not only nominal reward.

Good question to ask Mara:

> At Index, when learned policies fail on real hardware, is the bottleneck more often perception/representation, action prediction, data coverage, or hardware variability?

## Optional Extensions If Time Allows

- Add domain randomization during demonstration collection or policy training.
- Compare two policies: one trained clean, one trained with noisy observations.
- Save a short GIF of the clean policy and one failure mode.
- Repeat with `Pusher-v5` if you want a manipulation-flavored next step.